# Llama 3.1 8B SFT — 드라이빙 멘토링 AI 호스트

**실행 전 체크리스트**
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. 왼쪽 열솠 아이콘(Secrets) → `HF_TOKEN` 추가
3. 셀 순서대로 실행

In [ ]:
# 1. GPU 확인
import torch
assert torch.cuda.is_available(), 'GPU가 없습니다. 런타임 유형을 T4로 변경하세요.'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 2. 패키지 설치
!pip install -q \
    "transformers>=4.45.0" \
    "trl==1.5.1" \
    "peft>=0.13.0" \
    "accelerate>=0.33.0" \
    "bitsandbytes>=0.43.0" \
    "datasets>=2.20.0"
print('설치 완료')

In [ ]:
# 3. HuggingFace 인증
from huggingface_hub import login
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
    login(token=token, add_to_git_credential=False)
    print('Colab Secrets에서 HF_TOKEN 로드 성공')
except Exception:
    print('Secrets에 HF_TOKEN 없음 → 수동 입력')
    login()

In [ ]:
# 4. Google Drive 마운트
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/capstone'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive 경로: {DRIVE_DIR}')

In [ ]:
# 5. 학습 데이터 업로드
# 방법 A: 로컈 PC에서 직접 업로드
from google.colab import files
print('train_v2.jsonl 파일을 선택하세요')
uploaded = files.upload()
DATA_PATH = '/content/train_v2.jsonl'

# 방법 B: Drive에서 불러오기 (위 코드 대신 사용)
# DATA_PATH = f'{DRIVE_DIR}/train_v2.jsonl'

print(f'데이터 경로: {DATA_PATH}')

In [ ]:
# 6. 학습
import json
import pyarrow
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

BASE_MODEL     = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
OUTPUT_DIR     = '/content/adapters_v2'
LORA_RANK      = 8
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.05
TARGET_MODULES = ['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
BATCH_SIZE     = 1
GRAD_ACCUM     = 8
LEARNING_RATE  = 2e-4
NUM_EPOCHS     = 3
MAX_SEQ_LENGTH = 1024
SAVE_STEPS     = 100
LOGGING_STEPS  = 10

def load_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return Dataset.from_list(records)

def format_chat(example, tokenizer):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

print('[1/4] 토크나이저 로드...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
tokenizer.model_max_length = MAX_SEQ_LENGTH

print('[2/4] 데이터 로드...')
raw_dataset = load_jsonl(DATA_PATH)
dataset = raw_dataset.map(lambda ex: format_chat(ex, tokenizer))
print(f'  술 {len(dataset)}개 샘플')
sample_text = dataset[0]['text']
print(f'  예시: {sample_text[:120]}...')

print('[3/4] 모델 로드 (4-bit 양자화)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={'': 0},
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation='sdpa',
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print('[4/4] 학습 시작...')
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_steps=25,
    bf16=False,
    fp16=True,
    dataset_text_field='text',
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    report_to='none',
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

# LoRA adapter가 BF16으로 초기화됨 → FP32로 변환 (QLoRA 정석: adapter=FP32, compute=FP16)
for name, param in trainer.model.named_parameters():
    if param.requires_grad and param.dtype in (torch.bfloat16, torch.float16):
        param.data = param.data.to(torch.float32)

trainer.train()

print(f'\n학습 완료. 어댑터 저장 중 → {OUTPUT_DIR}')
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('저장 완료')

In [ ]:
# 7. Google Drive에 어댑터 백업
import shutil

dst = f'{DRIVE_DIR}/adapters_v2'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(OUTPUT_DIR, dst)
print(f'Drive 저장 완료: {dst}')